In [22]:
import pandas as pd 
import numpy as np
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
from torch.optim import AdamW

### Inspo
* https://www.kaggle.com/code/debarshichanda/bert-multi-label-text-classification

In [3]:
path = kagglehub.dataset_download("adisongoh/it-service-ticket-classification-dataset")

print("Path to dataset files:", path)

Path to dataset files: /home/james/.cache/kagglehub/datasets/adisongoh/it-service-ticket-classification-dataset/versions/1


In [4]:
df = pd.read_csv(f"{path}/all_tickets_processed_improved_v3.csv")
df.head()

,Document,Topic_group
0,connection with icon icon dear please setup ic...,Hardware
1,work experience user work experience user hi w...,Access
2,requesting for meeting requesting meeting hi p...,Hardware
3,reset passwords for external accounts re expir...,Access
4,mail verification warning hi has got attached ...,Miscellaneous


In [5]:
df["Topic_group"].value_counts()

Topic_group
Hardware                 13617
HR Support               10915
Access                    7125
Miscellaneous             7060
Storage                   2777
Purchase                  2464
Internal Project          2119
Administrative rights     1760
Name: count, dtype: int64

In [6]:
df.shape

(47837, 2)

In [7]:
df["text_length"] = df["Document"].apply(lambda x: len(x.split()))

In [8]:
df["text_length"].head()

0     18
1     19
2     14
3    145
4     15
Name: text_length, dtype: int64

In [9]:
df["text_length"].max()

np.int64(981)

In [10]:
df["label"] = LabelEncoder().fit_transform(df["Topic_group"])

### Decision
Longest text is 916 tokens, above the standard 512 token limit for BERT or Roberta. Our options are
* truncate text past 512 tokens
* split text into multiple records, then combine
* try a longer-attention option, like Longformer


For the base case, we'll truncate records to 512


In [11]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

### Dataset


In [45]:
class TicketDataset(Dataset):

    def __init__(self, data:list , labels:list , tokenizer, max_length=512):
        if not isinstance(data, list):
            raise ValueError(f"{type(data)} not of type list")
        
        self.data = data 
        self.labels = labels 
        self.tokenizer = tokenizer 
        self.max_length = max_length 

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        text = str(self.data[index])
        labels = self.labels[index]
        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True, 
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(labels, dtype=torch.long)
        }

In [43]:
X_train, X_test, train_y, test_y = train_test_split(df["Document"], df["label"], random_state=88, train_size=0.8)

In [ ]:
# Training config
EPOCHS = 20
BATCH_SIZE = 16
MAX_LENGTH = 512
LEARNING_RATE = 2e-5
N_CLASS = df["label"].nunique()

In [36]:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=N_CLASS, output_attentions=False, output_hidden_states=False)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [37]:
optimizer = AdamW(params=model.parameters(),  lr=LEARNING_RATE, weight_decay=1e-6)

In [38]:
for param in model.bert.parameters():
    param.requires_grad = False

In [39]:
for param in model.classifier.parameters():
    param.requires_grad = True

In [40]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [46]:
train_data = DataLoader(TicketDataset(X_train.to_list(), train_y.to_list(), tokenizer))

In [ ]:
total_steps = len(train_data) * EPOCHS

In [47]:
# Just for fun, am going to create a model
class TicketBert(nn.Module):

    def __init__(self):
        super().__init__(self)
        self.bert = model.bert 
        # self.fc = 

    def forward(self, X):
        pass

In [51]:
scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=.1 * total_steps,
    num_training_steps=total_steps|
)

SyntaxError: invalid syntax (2732742479.py, line 5)

In [ ]:

total_loss = 0
# Training
for i, batch in enumerate(train_data):

    # Move to device
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    # Forward pass
    outputs = model(
        input_ids = input_ids,
        attention_mask = attention_mask,
        labels=labels
    )
    loss = outputs.loss

    # Backward pass
    optimizer.zero_grad()
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()
    scheduler.step()


KeyboardInterrupt: 

In [49]:
def evaluate(model, test_dataloader):
    preds = [] 
    labels = []

    with torch.no_grad():
        for batch in test_dataloader:
            attention_mask = batch["attention_mask"].to(device)
            tokens= batch["input_ids"].to(device)
            labels = batch["labels"]

            outputs = model(tokens, attention_mask)

            labels.extend(list(batch["labels"]))
            preds.extend(outputs.logits.cpu().detach().numpy().to_list())

    return preds, labels

